In [ ]:
from pandas import offsets
import numpy as np
import pandas as pd
import scipy.ndimage as ndi

import tqdm
class MaskExtractor:
    def __init__(self, function=np.nanmean):
        # Contain directory of cell/unit id's based on centroid location
        self.centroid_to_id = {}
        self.next_id = 1
        self.results = []

        # Assign masking function (mean for ratio / intensity images and percentage for prediction images)
        self.func = function
    
    def _get_instance_id(self, centroid):
        """Function for identifying cell/unit id based on centroids"""
        if centroid not in self.centroid_to_id:
            self.centroid_to_id[centroid] = self.next_id
            self.next_id += 1
        return self.centroid_to_id[centroid]

    def _pad_obj_slice(self, obj_slice, dilations, image_shape):
        """Function for padding object slice""" 
        padded_slices = []
        insert_slices = []
        for coord_slice, max_len in zip(obj_slice, image_shape):
            pad_start = max(0, coord_slice.start - dilations)
            pad_end = min(max_len, coord_slice.stop + dilations)
            padded_slices.append(slice(pad_start, pad_end))
            
            insert_start = coord_slice.start - pad_start
            insert_stop = coord_slice.stop - pad_start
            insert_slices.append(slice(insert_start, insert_stop))
            
        return tuple(padded_slices), tuple(insert_slices)

    def _build_instance_metadata(self, mask_2d, dilations, min_instance_size=0):
        """Precompute labeled instances, padded slices, and centroids for a single 2D mask."""
        if mask_2d.ndim != 2:
            raise ValueError(f"Expected a 2D mask, got shape {mask_2d.shape}")

        labels, num_features = ndi.label(mask_2d)
        if num_features == 0:
            return None

        slices = ndi.find_objects(labels)
        instance_metadata = []

        for label_idx, slc in enumerate(slices, start=1):
            if slc is None:
                continue

            local_mask = labels[slc] == label_idx
            local_rows, local_cols = np.nonzero(local_mask)
            global_rows = local_rows + slc[0].start
            global_cols = local_cols + slc[1].start
            global_centroid = (int(np.mean(global_rows)), int(np.mean(global_cols)))
            instance_id = self._get_instance_id(global_centroid)

            padded_slc, insert_slc = self._pad_obj_slice(
                obj_slice=slc,
                dilations=dilations,
                image_shape=mask_2d.shape
            )

            padded_mask_shape = tuple(s.stop - s.start for s in padded_slc)
            padded_mask = np.zeros(padded_mask_shape, dtype=bool)
            padded_mask[insert_slc] = local_mask

            if dilations > 0:
                dilated_mask = ndi.binary_dilation(padded_mask, iterations=dilations)
            else:
                dilated_mask = padded_mask

            if min_instance_size and int(np.sum(dilated_mask)) < min_instance_size:
                continue

            instance_metadata.append({
                'instance_id': instance_id,
                'label_idx': label_idx,
                'padded_slice': padded_slc,
                'instance_mask': dilated_mask,
                'centroid_y': global_centroid[0],
                'centroid_x': global_centroid[1],
            })

        return instance_metadata

    def _process_instance_values(self, instance_metadata, val_2d, mask_name, val_name, metadata):
        """Quantify a single value image using precomputed instance metadata."""
        for inst in instance_metadata:
            val_mask = val_2d[inst['padded_slice']][inst['instance_mask']]
            val_value = self.func(val_mask) if val_mask.size > 0 else np.nan

            row_data = {
                'instance_id': inst['instance_id'],
                'mask_name': mask_name,
                'val_name': val_name,
                'centroid_y': inst['centroid_y'],
                'centroid_x': inst['centroid_x'],
                'value': val_value
            }
            row_data.update(metadata)
            self.results.append(row_data)

    def process_image(self, mask_image, value_image, mask_dictionary, value_dictionary, dilations=2, **metadata):
        """Global function for processing possible 3D masks and value images"""

        mask_metadata = mask_dictionary or {}
        value_metadata = value_dictionary or {}

        if mask_image.ndim == 2:
            num_mask_slices = 1
            mask_shape = mask_image.shape
        else:
            num_mask_slices = mask_image.shape[0]
            mask_shape = mask_image.shape[1:]
        
        if value_image.ndim == 2:
            num_value_slices = 1
            value_shape = value_image.shape
        else:
            num_value_slices = value_image.shape[0]
            value_shape = value_image.shape[1:]

        num_mask_slices = 1 if mask_image.ndim == 2 else mask_image.shape[0]
        num_value_slices = 1 if value_image.ndim == 2 else value_image.shape[0]

        # Resize mask if needed
        if mask_shape != value_shape:
            if mask_image.ndim == 2:
                zoom_factors = (value_shape[0] / mask_shape[0], value_shape[1] / mask_shape[1])
            if mask_image.ndim == 3:
                zoom_factors = (1, value_shape[0] / mask_shape[0], value_shape[1] / mask_shape[1] )
            mask_image = ndi.zoom(mask_image, zoom_factors, order=0)
                

        for mask_idx in range(num_mask_slices):
            mask_2d = mask_image if mask_image.ndim == 2 else mask_image[mask_idx]
            slice_mask_meta = mask_metadata.get(mask_idx, {})

            # Lookup name and fall back to index if nonexistent in dictionary
            m_name = slice_mask_meta.get('slice_name', f"Mask_{mask_idx}")
            
            instance_metadata = self._build_instance_metadata(
                mask_2d=mask_2d,
                dilations=dilations,
                min_instance_size=0
            )
            if not instance_metadata:
                continue
            
            for val_idx in range(num_value_slices):
                val_2d = value_image if value_image.ndim == 2 else value_image[val_idx]
                slice_value_meta = value_metadata.get(val_idx, {})

                # Lookup name again for value
                v_name = slice_value_meta.get('slice_name', f"Value_{val_idx}")

                # Combine all metadata into a single dictionary
                combined_metadata = {
                    **metadata,
                    **slice_mask_meta,
                    **slice_value_meta
                }

                # Process single 2D pair
                self._process_instance_values(
                    instance_metadata=instance_metadata,
                    val_2d=val_2d,
                    mask_name=m_name,
                    val_name=v_name,
                    metadata=combined_metadata
                )
    
    def return_dataframe(self):
        """Function for converting the aggregated results into a Pandas DataFrame"""
        return pd.DataFrame(self.results)

In [59]:
import tifffile
import zarr

def _lazy_load_image(file_path, res_level=0):
    """Function for loading standard .tif images and pyramidal ome-tiffs memory efficiently"""
    store = tifffile.imread(file_path, aszarr=True)
    z = zarr.open(store, mode='r')

    if isinstance(z, zarr.hierarchy.Group):
        # if pyramidal ome-tiff
        return z[str(res_level)]
    else:
        # if standard tif
        return z 

In [60]:
def _parse_mask_basename(basename, subgroups):
    """Return cleaned unit name and optional subgroup from a mask basename."""
    mask_name = basename

    # Split mask name into constitutive parts given pattern "<prefix>-<unit_name>-<mask.ext>"
    mask_parts = mask_name.split('-')
    unit_name = mask_parts[-2]

    # Convert to lower case to be case insensitive
    subgroups_lower = {k.lower(): v for k,v in subgroups.items()}

    subgroup = None
    if "_" in unit_name and subgroups_lower:
        unit_parts = unit_name.split("_")

        if len(unit_parts) == 2:
            # Handle the specific 2-part logic
            match0 = subgroups_lower.get(unit_parts[0].lower())
            match1 = subgroups_lower.get(unit_parts[1].lower())
            
            if match0:
                subgroup = match0  # Maps to the dictionary value
                unit_name = unit_parts[1]
            elif match1:
                subgroup = match1  # Maps to the dictionary value
                unit_name = unit_parts[0]
        else:
            for i, part in enumerate(unit_parts):
                match = subgroups_lower.get(part.lower())
                if match:
                    subgroup = match  # Maps to the dictionary value
                    remaining = [p for j, p in enumerate(unit_parts) if i != j]
                    unit_name = "_".join(remaining)
                    break

    return unit_name, subgroup

In [61]:
CONFIG = {
    'base_dir': r'C:\Users\jvill\OneDrive - UC San Diego\integrated_pipeline\WSI_data\WSI_data\Dataset_Root',
    
    'values_dir': 'values',
    'img_extensions' : ('.tif', '.ome.tiff', '.tiff'),

    'mask_dir': 'cluster_regions-masks',
    'subgroups' : {
      "C1": "Outer Cortex",
      "C2": "Middle Cortex",
      "C3": "Juxtamedullary Cortex",
      "CM": "Outer Stripe",
      "M": "Inner Stripe",
      "P": "Inner Medulla"
    },

    'int_dilations' : 2,
    'col_dilations' : 10,
}

MASK_DICTS = {
}

VAL_DICTS = {
  'Negative_Ion': {
    0: {'slice_name': "LPE 18:0 (-H)"},
    1: {'slice_name': "Unknown (N1)"},
    2: {'slice_name': "LPI 18:0 (-H)"},
    3: {'slice_name': "SM 18:1;2O/16:0 (-CH3)"},
    4: {'slice_name':"PA 36:1 (-H)"},
    5: {'slice_name':"PE 16:0_18:2 (-H)"},
    6: {'slice_name':"PE 16:0_18:0 (-H)"},
    7: {'slice_name':"PE 16:0_20:4 (-H)"},
    8: {'slice_name':"PE 16:0_22:6 (-H)"},
    9: {'slice_name':"PE 18:1_20:4 (-H)"},
    10: {'slice_name':"PE 18:1_20:3 (-H)"},
    11: {'slice_name':"PS 16:0_20:4 (-H)"},
    12: {'slice_name':"PS 18:0_18:1 (-H)"},
    13: {'slice_name':"PS 18:0_20:4 (-H)"},
    14: {'slice_name':"PG 18:1_22:6 (-H)"},
    15: {'slice_name':"HexCer 18:0;3O/23:0;(2OH) (-H)"},
    16: {'slice_name':"PI 34:1 (-H)"},
    17: {'slice_name':"PI 16:0_18:0 (-H)"},
    18: {'slice_name':"PI 16:0_20:4 (-H)"},
    19: {'slice_name':"PI 18:0_18:2 (-H)"},
    20: {'slice_name':"PI 18:0_18:1 (-H)"},
    21: {'slice_name':"SHexCer 40:1;3O (-H)"},
    22: {'slice_name':"PI 18:1_20:3 (-H)"},
    23: {'slice_name':"SHexCer 41:1;3O (-H)"},
    24: {'slice_name':"SM(d34:1) (+DMACA)"},
    25: {'slice_name':"PC 46:9 (-CH3)"},
    26: {'slice_name':"SHexCer 42:2;3O (-H)"},
    27: {'slice_name':"SHexCer 42:1;3O (-H)"},
    28: {'slice_name':"SHexCer 42:0;3O (-H)"},
    29: {'slice_name':"PE 49:9 (-H)"},
    30: {'slice_name':"PC 48:9 (-CH3)"},
    31: {'slice_name':"SHexCer 50:0;2O (-H)"},
    32: {'slice_name':"CL 72:8 (-H)"},
    33: {'slice_name':"CL 74:11 (-H)"},

  },
  'Positive_Ion': {
    0: {'slice_name': "Unknown (P1)"},
    1: {'slice_name': "Unknown (P2)"},
    2: {'slice_name':"CAR 18:2 (+H)"},
    3: {'slice_name':"CAR 18:1 (+H)"},
    4: {'slice_name':"CAR 18:0 (+H)"},
    5: {'slice_name':"Cer 17:2;2O/8:0;O (+Na)"},
    6: {'slice_name':"LPC 16:0 (+H)"},
    7: {'slice_name':"Unknown (P3)"},
    8: {'slice_name':"SM 34:1;2O (+H)"},
    9: {'slice_name':"Unknown (P4)"},
    10: {'slice_name':"PC 32:0 (+H)"},
    11: {'slice_name':"PC 34:2 (+H)"},
    12: {'slice_name':"SM 38:1;20 (+H)"},
    13: {'slice_name':"PC 16:0_18:1 (+H)"},
    14: {'slice_name':"PC 35:4 (+H)"},
    15: {'slice_name':"PC 36:5 (+H)"},
    16: {'slice_name':"PC 36:3 (+H)"},
    17: {'slice_name':"PC 18:0_18:1 (+H)"},
    18: {'slice_name':"PC O-38:5 (+H)"},
    19: {'slice_name':"PC 38:5 (+H)"},
    20: {'slice_name':"PC 18:0_20:4 (+H)"},
    21: {'slice_name':"SM 42:2;2O (+H)"},
    22: {'slice_name':"SM 42:1;2O (+H)"},
    23: {'slice_name':"SM 41:0;3O (+H)"},
    24: {'slice_name':"SM 42:0;3O (+H)}"}
  },

}

In [62]:
from pathlib import Path
extractor = MaskExtractor()
data_root = Path(CONFIG['base_dir'])  

for i, sample_dir in enumerate(data_root.iterdir()):
    if not sample_dir.is_dir():
        continue
    if i == 0:
        continue

    tissue_id = sample_dir.name
    masks_dir = sample_dir / CONFIG['mask_dir']
    values_dir = sample_dir / CONFIG['values_dir']

    mask_files = [m_file for m_file in masks_dir.glob('*.*tif*')
    if m_file.name.lower().endswith(CONFIG['img_extensions'])]

    val_files = [v_file for v_file in values_dir.glob('*.*tif*')
    if v_file.name.lower().endswith(CONFIG['img_extensions'])]

    value_images = []
    for value_file in val_files:
        value_lazy = _lazy_load_image(value_file, res_level=0)
        val_basename = value_file.name.split('.')[0]
        val_name = val_basename.split('-')[-2]

        print(f"Val basename: {val_basename}\nVal name: {val_name}")

        if val_name not in VAL_DICTS and value_lazy.ndim == 2:
            VAL_DICTS[val_name] = {
                0: {
                    'slice_name': val_name
                }
            }

        if "Collagen" in val_basename:
            dilations = CONFIG['col_dilations']
        else:
            dilations = CONFIG['int_dilations']

        value_images.append({
            'value_lazy': value_lazy,
            'val_basename': val_basename,
            'val_name': val_name,
            'dilations': dilations
        })

    for mask_file in tqdm.tqdm(mask_files[:5], desc="Processing masks..."):
        mask_image = tifffile.imread(mask_file) > 0
        mask_basename = mask_file.name.split('.')[0]
        mask_name = mask_basename.split('-')[-2]

        print(f"Mask basename: {mask_basename}\nMask name: {mask_name}")

        if mask_name not in MASK_DICTS and mask_image.ndim == 2: 
            unit_name, subgroup = _parse_mask_basename(mask_basename, CONFIG['subgroups'])
            print(f"Unit name: {unit_name}")
            print(f"Group: {subgroup}")
            MASK_DICTS[mask_name] = {
                0 : {
                    'slice_name': unit_name,
                    "Group" : subgroup
                }
            }

        for cached_value in value_images:
            value_lazy = cached_value['value_lazy']
            val_basename = cached_value['val_basename']
            val_name = cached_value['val_name']
            dilations = cached_value['dilations']

            extractor.process_image(
                mask_image=mask_image,
                value_image=value_lazy,
                mask_dictionary=MASK_DICTS[mask_name],
                value_dictionary=VAL_DICTS[val_name],
                dilations=dilations,
                tissue_id=tissue_id
            )


Val basename: S24-K21_273_3-Collagen_Coherence-collagen
Val name: Collagen_Coherence
Val basename: S24-K21_273_3-Collagen_Thickness-collagen
Val name: Collagen_Thickness
Val basename: S24-K21_273_3-Lipid_to_Protein-ratio
Val name: Lipid_to_Protein
Val basename: S24-K21_273_3-Lipid_Unsaturation-ratio
Val name: Lipid_Unsaturation
Val basename: S24-K21_273_3-Negative_Ion-ratio
Val name: Negative_Ion
Val basename: S24-K21_273_3-Positive_Ion-ratio
Val name: Positive_Ion


Processing masks...:   0%|          | 0/5 [00:00<?, ?it/s]

Mask basename: CODEX-Cluster10_C3-mask
Mask name: Cluster10_C3
Unit name: Cluster10
Group: Juxtamedullary Cortex


Processing masks...:  20%|██        | 1/5 [00:17<01:09, 17.37s/it]

Mask basename: CODEX-Cluster10_CM-mask
Mask name: Cluster10_CM
Unit name: Cluster10
Group: Outer Stripe


...

Processing masks...:  60%|██████    | 3/5 [00:51<00:34, 17.04s/it]

Mask basename: CODEX-Cluster10_P-mask
Mask name: Cluster10_P
Unit name: Cluster10
Group: Inner Medulla


Processing masks...:  80%|████████  | 4/5 [01:08<00:17, 17.18s/it]

Mask basename: CODEX-Cluster11_C3-mask
Mask name: Cluster11_C3
Unit name: Cluster11
Group: Juxtamedullary Cortex


Processing masks...: 100%|██████████| 5/5 [01:27<00:00, 17.50s/it]


In [63]:
df = extractor.return_dataframe()

In [64]:
df.head()

,instance_id,mask_name,val_name,centroid_y,centroid_x,value,tissue_id,slice_name,Group
0,1,Cluster10,Collagen_Coherence,1270,8315,0.000000,S24-K21_273_3,Collagen_Coherence,Juxtamedullary Cortex
1,2,Cluster10,Collagen_Coherence,1925,8796,0.000835,S24-K21_273_3,Collagen_Coherence,Juxtamedullary Cortex
2,3,Cluster10,Collagen_Coherence,2016,8361,0.158021,S24-K21_273_3,Collagen_Coherence,Juxtamedullary Cortex
3,4,Cluster10,Collagen_Coherence,2111,8970,0.040248,S24-K21_273_3,Collagen_Coherence,Juxtamedullary Cortex
4,5,Cluster10,Collagen_Thickness,2540,16630,0.007992,S24-K21_273_3,Collagen_Thickness,Juxtamedullary Cortex


In [65]:
feature_matrix = df.pivot(
    index=['instance_id', 'tissue_id', 'mask_name', 'centroid_y', 'centroid_x'], 
    columns='val_name', 
    values='value'
).reset_index()

In [66]:
feature_matrix

val_name,instance_id,tissue_id,mask_name,centroid_y,centroid_x,CAR 18:0 (+H),CAR 18:1 (+H),CAR 18:2 (+H),CL 72:8 (-H),CL 74:11 (-H),...,SM 41:0;3O (+H),SM 42:0;3O (+H)},SM 42:1;2O (+H),SM 42:2;2O (+H),SM(d34:1) (+DMACA),Unknown (N1),Unknown (P1),Unknown (P2),Unknown (P3),Unknown (P4)
0,1,S24-K21_273_3,Cluster10,1270,8315,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,S24-K21_273_3,Cluster10,1925,8796,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,S24-K21_273_3,Cluster10,2016,8361,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,S24-K21_273_3,Cluster10,2111,8970,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,S24-K21_273_3,Cluster10,2540,16630,0.0,0.0,0.0,110.442308,15.990385,...,0.0,0.0,0.0,0.0,13.692308,6.836538,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
853,854,S24-K21_273_3,Cluster11,3610,17015,0.0,0.0,0.0,47.037657,7.895397,...,0.0,0.0,0.0,0.0,22.225941,21.213389,0.0,0.0,0.0,0.0
854,855,S24-K21_273_3,Cluster11,3678,17727,0.0,0.0,0.0,81.125628,18.226131,...,0.0,0.0,0.0,0.0,8.376884,0.050251,0.0,0.0,0.0,0.0
855,856,S24-K21_273_3,Cluster11,3686,17740,0.0,0.0,0.0,56.997245,33.068871,...,0.0,0.0,0.0,0.0,13.812672,5.842975,0.0,0.0,0.0,0.0
856,857,S24-K21_273_3,Cluster11,3696,17725,0.0,0.0,0.0,59.335165,35.351648,...,0.0,0.0,0.0,0.0,9.505495,2.412088,0.0,0.0,0.0,0.0


In [67]:
feature_matrix.columns

Index(['instance_id', 'tissue_id', 'mask_name', 'centroid_y', 'centroid_x',
       'CAR 18:0 (+H)', 'CAR 18:1 (+H)', 'CAR 18:2 (+H)', 'CL 72:8 (-H)',
       'CL 74:11 (-H)', 'Cer 17:2;2O/8:0;O (+Na)', 'Collagen_Coherence',
       'Collagen_Thickness', 'HexCer 18:0;3O/23:0;(2OH) (-H)', 'LPC 16:0 (+H)',
       'LPE 18:0 (-H)', 'LPI 18:0 (-H)', 'Lipid_Unsaturation',
       'Lipid_to_Protein', 'PA 36:1 (-H)', 'PC 16:0_18:1 (+H)',
       'PC 18:0_18:1 (+H)', 'PC 18:0_20:4 (+H)', 'PC 32:0 (+H)',
       'PC 34:2 (+H)', 'PC 35:4 (+H)', 'PC 36:3 (+H)', 'PC 36:5 (+H)',
       'PC 38:5 (+H)', 'PC 46:9 (-CH3)', 'PC 48:9 (-CH3)', 'PC O-38:5 (+H)',
       'PE 16:0_18:0 (-H)', 'PE 16:0_18:2 (-H)', 'PE 16:0_20:4 (-H)',
       'PE 16:0_22:6 (-H)', 'PE 18:1_20:3 (-H)', 'PE 18:1_20:4 (-H)',
       'PE 49:9 (-H)', 'PG 18:1_22:6 (-H)', 'PI 16:0_18:0 (-H)',
       'PI 16:0_20:4 (-H)', 'PI 18:0_18:1 (-H)', 'PI 18:0_18:2 (-H)',
       'PI 18:1_20:3 (-H)', 'PI 34:1 (-H)', 'PS 16:0_20:4 (-H)',
       'PS 18:0_1